In [ ]:
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "jupytergis==0.16.8",
#     "openeo",
# ]
# ///

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and then launch this notebook with:

```
uvx juv run snow_classifier_sentinel2.ipynb
```

(The `uvx` command is included when installing `uv`).

## License

This notebook is an adaptation of the openEO-UDP notebook
[snow_classifier.ipynb](https://github.com/developmentseed/openeo-udp/blob/main/notebooks/sentinel/sentinel-2/land-cover/snow_classifier.ipynb)
by Development Seed. That notebook is a conversion of a Sentinel Hub evalscript and is licensed under **CC-BY-SA-4.0**.
This adaptation is also licensed under **CC-BY-SA-4.0**.

- Original evalscript: https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/snow_classifier/
- Source: Sentinel Hub Custom Scripts (CC-BY-SA-4.0)
- Conversion: Development Seed (openEO-UDP project)

# Snow classification around Bovec, Slovenia, October 2019

This notebook detects snow in the Julian Alps around Bovec (Slovenia) from Sentinel-2 L2A data with openEO.
It shows the result as a dynamic tile layer in a JupyterGIS map, on top of an OpenStreetMap basemap.

The classifier uses the Normalized Difference Snow Index (NDSI) to detect snow, and the Normalized Difference Vegetation Index (NDVI) as an additional constraint:

$$
NDSI = \frac{B03 - B11}{B03 + B11}
\qquad
NDVI = \frac{B08 - B04}{B08 + B04}
$$

Snow is bright in the visible (B03) and dark in the SWIR (B11), so it has a high NDSI.
Clouds are bright in both bands, so NDSI separates snow from most clouds.

The steps are:
1. Connect to the openEO backend.
2. Load the Sentinel-2 L2A scenes of Bovec from 26 to 30 October 2019 and make a first-valid-pixel mosaic.
3. Calculate NDSI and NDVI and classify each pixel as snow or not snow.
4. Show snow pixels in a light beige color and the other pixels as a brightened true color composite.
5. Show the result in JupyterGIS.

## Connect to the openEO backend

In [ ]:
import openeo
from openeo.processes import absolute, and_, array_create, if_, or_

connection = openeo.connect("https://openeo.mpc.ds.io/")

In [ ]:
connection.authenticate_oidc()

## Parameters

The bounding box covers Bovec and the surrounding peaks of the Julian Alps (Kanin, Krn).

The time range is 26 to 30 October 2019. The end date is exclusive in openEO.
The cloud cover filter keeps only scenes with 30 % cloud cover or less.

`ndsi_threshold` is the NDSI value above which a pixel is classified as snow.

In [ ]:
location_name = "Bovec, Slovenia"

bounding_box = {"west": 13.40, "south": 46.32, "east": 13.54, "north": 46.39}
temporal_extent = ["2019-10-26", "2019-10-30"]
max_cloud_cover = 30
ndsi_threshold = 0.4

# Sentinel-2 bands for the algorithm, in this order:
# B02 (blue), B03 (green), B04 (red), B08 (NIR), B11 (SWIR 1610 nm)
bands = ["B02", "B03", "B04", "B08", "B11"]

## Load the Sentinel-2 data

The algorithm needs the visible bands for the true color composite, and the green, red, NIR and SWIR bands for the NDSI and NDVI.

- **B02**: Blue, 492.4 nm (true color)
- **B03**: Green, 559.8 nm (NDSI, true color)
- **B04**: Red, 664.6 nm (NDVI, true color)
- **B08**: NIR, 832.8 nm (NDVI)
- **B11**: SWIR, 1613.7 nm (NDSI)

In [ ]:
s2cube = connection.load_collection(
    "sentinel-2-l2a",
    spatial_extent=bounding_box,
    temporal_extent=temporal_extent,
    bands=bands,
    properties={"eo:cloud_cover": lambda x: x <= max_cloud_cover},
)

s2cube = s2cube.process(
    "apply_pixel_selection",
    pixel_selection="first",
    data=s2cube,
)

## Snow classification

A pixel is classified as snow when both conditions are true:

1. `NDSI >= ndsi_threshold` (0.4), **or** NDVI is close to 0.1 (`|NDVI - 0.1| <= 0.025`). The NDVI condition catches snow with a lower NDSI, for example snow mixed with sparse vegetation.
2. The pixel is bright in the green band (`B03 > 0.3`). This removes dark pixels such as water and shadows, which can also have a high NDSI.

| Class | Color |
|---|---|
| Snow | Light beige (R=1.0, G=0.8, B=0.4) |
| Not snow | True color (B04, B03, B02), brightened with a gain of 2.5 |

The backend returns reflectance values from 0 to 1, so the bands are used without scaling.

In [ ]:
gain = 2.5  # brightness gain for the true color composite


def snow_classifier(data):
    """
    Classify snow in Sentinel-2 L2A imagery with NDSI and NDVI.
    Input data array: [B02, B03, B04, B08, B11]

    Snow pixels are shown in light beige.
    Other pixels are shown as a brightened true color composite.
    """
    B02, B03, B04, B08, B11 = data[0], data[1], data[2], data[3], data[4]

    NDSI = (B03 - B11) / (B03 + B11)
    NDVI = (B08 - B04) / (B08 + B04)

    # Snow: high NDSI, or NDVI close to 0.1
    is_snow = or_(NDSI >= ndsi_threshold, absolute(NDVI - 0.1) <= 0.025)

    # Snow must also be bright in the green band
    snow_mask = and_(is_snow, B03 > 0.3)

    red = if_(snow_mask, 1.0, B04 * gain)
    green = if_(snow_mask, 0.8, B03 * gain)
    blue = if_(snow_mask, 0.4, B02 * gain)

    return array_create([red, green, blue])


# Apply the snow classifier on the data cube on the bands dimension
snow_image = s2cube.apply_dimension(
    dimension=s2cube.metadata.band_dimension.name, process=snow_classifier
)

# Linear scale to 0-255 for RGB visualization
snow_image = snow_image.linear_scale_range(
    input_min=0, input_max=1, output_min=0, output_max=255
)

graph = snow_image

## Show the result in JupyterGIS

The map has an OpenStreetMap basemap. The openEO layer is added on top of it.
`add_openeo_tile_layer` creates an XYZ service on the backend and fits the map to the bounding box.

The backend calculates each tile on request. A tile can take 10 to 25 seconds, so the layer fills in slowly.

Light beige is snow. The other pixels are true color.

In [ ]:
from jupytergis import GISDocument

doc = GISDocument(latitude=46.355, longitude=13.47, zoom=12)

doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
    attribution="© OpenStreetMap contributors",
)

In [ ]:
doc.add_openeo_tile_layer(graph, name="Snow Classifier")
doc

## Technical notes

**Why NDSI for snow detection?**

- Snow reflects strongly in the visible bands (B03) and absorbs strongly in the SWIR (B11), so its NDSI is high.
- Clouds reflect strongly in both the visible and the SWIR, so their NDSI is low. NDSI separates snow from most clouds, which a true color image cannot do.
- Water and shadows can have a high NDSI too, but they are dark. The `B03 > 0.3` brightness check removes them.

**Thresholds:**
- The default `ndsi_threshold` of 0.4 is a common value in the literature for Sentinel-2 and Landsat.
- Lower the threshold to detect thin or patchy snow. Raise it to reduce false positives over bright soil or rock.

## Attribution

**Original script:** [Snow Classifier](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/snow_classifier/) (authors not listed), [Sentinel Hub Custom Scripts](https://github.com/sentinel-hub/custom-scripts)

**Citation:**
- Hagolle, O., Huc, M., Desjardins, C., Auer, S., Richter, R. (2017). MAJA Algorithm Theoretical Basis Document (Version 1.0). Zenodo. https://doi.org/10.5281/zenodo.1209633
- Dietz, A.J., Kuenzer, C., Gessner, U., Dech, S. (2012). Remote Sensing of Snow – A Review of Available Methods. International Journal of Remote Sensing, 33(13), 4094–4134. https://dx.doi.org/10.1080/01431161.2011.640964